# 🏆 Mission 2 — 다중 모델 벤치마크 & 하이브리드 앙상블

대회 공식 가이드에 명시된 **고전 머신러닝(RandomForest)**, **2D 비전 CNN(ResNet50 1ch / 3ch)**, **음성 특화 트랜스포머(Wav2Vec2)** 등
다양한 패러다임을 비교 실험하고, 최종 **하이브리드 앙상블**을 통해 최고 성능을 달성합니다.

| 단계 | 모델 | 입력 형태 |
|---|---|---|
| 경로 A | RandomForest | MFCC 통계치 (1D 벡터) |
| 경로 B-1 | ResNet50 (1ch) | Mel-Spectrogram 흑백 |
| 경로 B-2 | ResNet50 (3ch Fusion) | Mel + MFCC + Chroma 컬러 |
| 경로 C | Wav2Vec 2.0 | Raw Waveform (1D 파형) |
| 최종 | 하이브리드 앙상블 | 위 모델들의 확률 결합 |

In [5]:
# ==========================================
# 0-1. 구글 드라이브 마운트 및 전체 분할 압축 해제 (001~013 전체)
# ==========================================
import os, glob, subprocess
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/코찔찔이/DCC'
EXTRACT_DIR = '/content/dataset'
os.makedirs(EXTRACT_DIR, exist_ok=True)

# 2. 드라이브 내 압축 파일 목록 확인 (.zip 파일 전체)
zip_files = sorted(glob.glob(f'{DRIVE_DIR}/*.zip'))
print(f'>> 발견된 zip 파일 총 {len(zip_files)}개:')
for zf in zip_files:
    print('  -', os.path.basename(zf))

# 3. p7zip 도구 설치
!apt-get install -y p7zip-full -q

# 4. 001부터 013까지 모든 zip 파일 순차 해제
# (구글 드라이브 다운로드 파일은 각각 다른 파일들을 나눠 담고 있으므로 13개 모두 풀어야 완성됩니다)
print(f'\n>> 총 {len(zip_files)}개 파일 순차 압축 해제 시작 (/content/dataset)...')
for i, zf in enumerate(zip_files, 1):
    fname = os.path.basename(zf)
    print(f'[{i}/{len(zip_files)}] 압축 푸는 중: {fname} ...', flush=True)
    !7z x "{zf}" -o{EXTRACT_DIR} -y > /dev/null

print('\n>> 모든 압축 해제 완료!')
print('추출된 상위 폴더 목록:')
!ls -la {EXTRACT_DIR}



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
>> 발견된 zip 파일 총 13개:
  - 대학부 데이터-20260830T163301Z-1-001.zip
  - 대학부 데이터-20260830T163301Z-1-002.zip
  - 대학부 데이터-20260830T163301Z-1-003.zip
  - 대학부 데이터-20260830T163301Z-1-004.zip
  - 대학부 데이터-20260830T163301Z-1-005.zip
  - 대학부 데이터-20260830T163301Z-1-006.zip
  - 대학부 데이터-20260830T163301Z-1-007.zip
  - 대학부 데이터-20260830T163301Z-1-008.zip
  - 대학부 데이터-20260830T163301Z-1-009.zip
  - 대학부 데이터-20260830T163301Z-1-010.zip
  - 대학부 데이터-20260830T163301Z-1-011.zip
  - 대학부 데이터-20260830T163301Z-1-012.zip
  - 대학부 데이터-20260830T163301Z-1-013.zip
Reading package lists...
Building dependency tree...
Reading state information...
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.

>> 총 13개 파일 순차 압축 해제 시작 (/content/dataset)...
[1

In [ ]:
# ==========================================
# 0-2. 데이터 경로 자동 감지 및 스마트 연결
# ==========================================
import os, sys, shutil

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    os.chdir('/content')
    
    # 1. 압축 풀린 경로에서 train과 val 폴더 스마트 자동 탐색
    print(">> 데이터셋 폴더 구조 탐색 중...")
    train_dir, val_dir = None, None
    
    for root, dirs, files in os.walk('/content/dataset'):
        for d in dirs:
            full_path = os.path.join(root, d)
            d_lower = d.lower()
            
            # train 탐색 (내부에 label 또는 audio 폴더가 있는 폴더)
            if ('train' in d_lower) and not train_dir:
                subdirs = os.listdir(full_path)
                if any(s.lower() in ['audio', 'label'] for s in subdirs) or any(f.endswith(('.wav', '.json')) for f in os.listdir(full_path)[:10]):
                    train_dir = full_path
                    
            # val 탐색 (내부에 label 또는 audio 폴더가 있는 폴더)
            if ('val' in d_lower or 'valid' in d_lower) and not val_dir:
                subdirs = os.listdir(full_path)
                if any(s.lower() in ['audio', 'label'] for s in subdirs) or any(f.endswith(('.wav', '.json')) for f in os.listdir(full_path)[:10]):
                    val_dir = full_path
                    
        if train_dir and val_dir:
            break

    print(f"  * 감지된 train 경로: {train_dir}")
    print(f"  * 감지된 val   경로: {val_dir}")
    
    # 2. /content/data 심볼릭 링크 생성
    os.makedirs('/content/data', exist_ok=True)
    
    if train_dir:
        target_train = '/content/data/train'
        if os.path.exists(target_train):
            os.remove(target_train) if os.path.islink(target_train) else shutil.rmtree(target_train)
        os.symlink(train_dir, target_train)
        print("  -> /content/data/train 연결 완료!")
    else:
        print("  [경고] train 폴더를 찾지 못했습니다. /content/dataset 내부 구조를 확인하세요.")
        
    if val_dir:
        target_val = '/content/data/val'
        if os.path.exists(target_val):
            os.remove(target_val) if os.path.islink(target_val) else shutil.rmtree(target_val)
        os.symlink(val_dir, target_val)
        print("  -> /content/data/val 연결 완료!")
    else:
        print("  [경고] val 폴더를 찾지 못했습니다. /content/dataset 내부 구조를 확인하세요.")

    # 3. 필수 패키지 설치
    print("\n>> 필수 패키지 설치 중...")
    !pip install -q librosa transformers datasets evaluate soundfile
    print("GPU 정보 확인:")
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else:
    if os.path.basename(os.getcwd()) == 'mission2_speaker':
        os.chdir('..')
    print("로컬 환경에서 실행 중입니다.")

print(f"\n현재 작업 디렉토리: {os.getcwd()}")
train_exists = os.path.exists('data/train')
val_exists = os.path.exists('data/val')
print(f"데이터 연동 결과: train={train_exists}, val={val_exists}")

if train_exists:
    label_p = 'data/train/label'
    audio_p = 'data/train/audio'
    if os.path.exists(label_p):
        print(f"  - Train JSON 라벨 수: {len(os.listdir(label_p)):,}개")
    if os.path.exists(audio_p):
        print(f"  - Train WAV 오디오 수: {len(os.listdir(audio_p)):,}개")

if val_exists:
    label_p = 'data/val/label'
    audio_p = 'data/val/audio'
    if os.path.exists(label_p):
        print(f"  - Val JSON 라벨 수  : {len(os.listdir(label_p)):,}개")
    if os.path.exists(audio_p):
        print(f"  - Val WAV 오디오 수  : {len(os.listdir(audio_p)):,}개")



In [3]:
# ==========================================
# 1. 공용 라이브러리 임포트 및 GPU/CPU 디바이스 설정
# ==========================================
import os
import json
import random
import time
import numpy as np
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# GPU/CPU 자동 감지
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 중인 디바이스: {device}')

# 재현성을 위한 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 데이터 경로 설정
TRAIN_LABEL_DIR = 'data/train/label'
TRAIN_AUDIO_DIR = 'data/train/audio'
VAL_LABEL_DIR   = 'data/val/label'
VAL_AUDIO_DIR   = 'data/val/audio'

SR = 16000           # 샘플링 레이트
MAX_TIME_STEPS = 256 # 스펙트로그램 가로 길이 (약 3초)
N_MELS = 128         # Mel 필터뱅크 개수
BATCH_SIZE = 32

사용 중인 디바이스: cuda


In [4]:
# ==========================================
# 2. 공용 데이터 파싱 (모든 모델이 공유하는 발화 조각 목록)
# ==========================================
def parse_utterances(label_dir, audio_dir):
    """
    라벨(JSON) 폴더와 오디오(WAV) 폴더를 읽어서
    [wav경로, 시작ms, 종료ms, 화자(0/1)] 목록을 반환합니다.
    대회 규칙: startAt, endAt 외의 annotation 사용 불가!
    """
    samples = []
    for json_name in sorted(os.listdir(label_dir)):
        if not json_name.endswith('.json'):
            continue
        json_path = os.path.join(label_dir, json_name)
        wav_name = json_name.replace('.json', '.wav')
        wav_path = os.path.join(audio_dir, wav_name)
        if not os.path.exists(wav_path):
            continue
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        for utt in data.get('utterances', []):
            samples.append({
                'wav_path': wav_path,
                'wav_name': wav_name,
                'start_ms': utt['startAt'],
                'end_ms':   utt['endAt'],
                'label':    utt['speaker']   # 1: 신고자, 0: 119대원
            })
    return samples

print("발화 조각 목록 파싱 중...")
train_samples = parse_utterances(TRAIN_LABEL_DIR, TRAIN_AUDIO_DIR)
val_samples   = parse_utterances(VAL_LABEL_DIR,   VAL_AUDIO_DIR)
print(f"  Train: {len(train_samples):,}개 조각")
print(f"  Val:   {len(val_samples):,}개 조각")

발화 조각 목록 파싱 중...


FileNotFoundError: [Errno 2] No such file or directory: 'data/train/label'

In [ ]:
# ==========================================
# 3. 공용 오디오 로딩 유틸리티 (캐싱 포함)
# ==========================================
# 같은 wav 파일을 반복 디코딩하지 않도록 파일 단위 캐시
_audio_cache = {}

def load_audio_cached(wav_path, sr=SR):
    """wav 파일을 캐시하여 같은 파일 재디코딩을 방지합니다."""
    if wav_path not in _audio_cache:
        y, _ = librosa.load(wav_path, sr=sr)
        _audio_cache[wav_path] = y
    return _audio_cache[wav_path]

def cut_audio(sample, sr=SR):
    """발화 조각의 시작/끝 시간(ms)에 맞춰 오디오를 잘라냅니다."""
    y = load_audio_cached(sample['wav_path'], sr=sr)
    s = int((sample['start_ms'] / 1000.0) * sr)
    e = int((sample['end_ms']   / 1000.0) * sr)
    audio_cut = y[s:e]
    # 너무 짧은 조각 방어 (최소 0.1초)
    if len(audio_cut) < int(0.1 * sr):
        audio_cut = np.zeros(int(0.1 * sr))
    return audio_cut

print("오디오 유틸리티 준비 완료!")

---
# 🌲 경로 A: 고전 머신러닝 — RandomForest

딥러닝 없이 **MFCC 통계치(평균/표준편차)**만으로 도달할 수 있는 정확도의 하한선을 측정합니다.
이 결과가 딥러닝 모델들의 성능이 '정말 의미 있는 것인지' 판단하는 기준선(Baseline)이 됩니다.

In [ ]:
# ==========================================
# [경로 A] RandomForest용 피처 추출
# ==========================================
def extract_rf_features(sample, sr=SR):
    """
    오디오 조각에서 RandomForest에 먹일 통계적 특징 벡터를 추출합니다.
    - MFCC 20개 계수의 평균 & 표준편차 (40차원)
    - Spectral Centroid 평균 (1차원): 소리의 '밝기/무게중심'
    - Zero Crossing Rate 평균 (1차원): 무음/잡음 비율 지표
    - RMS Energy 평균 (1차원): 소리의 크기(데시벨)
    총 43차원 벡터
    """
    audio = cut_audio(sample, sr=sr)
    
    # MFCC: 사람 음성의 음색(톤)을 숫자 20개로 요약
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    mfcc_mean = np.mean(mfcc, axis=1)    # 각 계수의 평균 (20개)
    mfcc_std  = np.std(mfcc, axis=1)     # 각 계수의 편차 (20개)
    
    # Spectral Centroid: 주파수의 무게중심 (높으면 밝은 소리)
    centroid = np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr))
    
    # Zero Crossing Rate: 파형이 0을 지나는 횟수 (잡음/무음 판별)
    zcr = np.mean(librosa.feature.zero_crossing_rate(y=audio))
    
    # RMS Energy: 소리의 전체적인 크기
    rms = np.mean(librosa.feature.rms(y=audio))
    
    return np.concatenate([mfcc_mean, mfcc_std, [centroid, zcr, rms]])

print("RandomForest 피처 추출 함수 준비 완료!")
print("피처 벡터 차원:", extract_rf_features(train_samples[0]).shape[0])

In [ ]:
# ==========================================
# [경로 A] RandomForest 학습 및 평가
# ==========================================
print("Train 피처 추출 중... (1~2분 소요)")
t0 = time.time()
X_train = np.array([extract_rf_features(s) for s in tqdm(train_samples, desc="Train 피처")])
y_train = np.array([s['label'] for s in train_samples])
print(f"  Train 피처 추출 완료! ({time.time()-t0:.1f}초)")

print("Val 피처 추출 중...")
t0 = time.time()
X_val = np.array([extract_rf_features(s) for s in tqdm(val_samples, desc="Val 피처")])
y_val = np.array([s['label'] for s in val_samples])
print(f"  Val 피처 추출 완료! ({time.time()-t0:.1f}초)")

# RandomForest 학습 (나무 200그루, CPU로 1분 내 완료)
print("\nRandomForest 학습 시작...")
t0 = time.time()
rf_model = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=SEED, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_train_time = time.time() - t0
print(f"  학습 완료! ({rf_train_time:.1f}초)")

# 평가
rf_preds = rf_model.predict(X_val)
rf_accuracy = accuracy_score(y_val, rf_preds) * 100
print(f"\n{'='*50}")
print(f"[경로 A] RandomForest 검증 정확도: {rf_accuracy:.2f}%")
print(f"{'='*50}")
print(classification_report(y_val, rf_preds, target_names=['119대원(0)', '신고자(1)']))

---
# 👁️ 경로 B-1: ResNet50 1채널 (기존 베이스라인)

현재 구축된 기존 모델과 동일한 구조입니다.
Mel-Spectrogram **흑백 1채널**을 1채널로 개조한 ResNet50에 입력합니다.

In [ ]:
# ==========================================
# [경로 B] 공용 스펙트로그램 Dataset 클래스
# ==========================================
class SpeakerDataset(Dataset):
    """
    n_channels=1 이면 기존 Mel 흑백 (경로 B-1)
    n_channels=3 이면 Mel+MFCC+Chroma 컬러 퓨전 (경로 B-2)
    """
    def __init__(self, samples, n_channels=1, max_time_steps=MAX_TIME_STEPS, 
                 sr=SR, n_mels=N_MELS, is_train=True):
        self.samples = samples
        self.n_channels = n_channels
        self.max_time_steps = max_time_steps
        self.sr = sr
        self.n_mels = n_mels
        self.is_train = is_train
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        audio = cut_audio(sample, sr=self.sr)
        
        # 채널 1: Mel-Spectrogram (모든 경로 공용)
        mel = librosa.feature.melspectrogram(y=audio, sr=self.sr, n_mels=self.n_mels)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        
        if self.n_channels == 3:
            # 채널 2: MFCC (음색/톤 특징) — n_mels 개로 맞추기 위해 resize
            mfcc = librosa.feature.mfcc(y=audio, sr=self.sr, n_mfcc=self.n_mels)
            
            # 채널 3: Chroma STFT (음계/피치 특징) — 12 -> n_mels 로 resize
            chroma = librosa.feature.chroma_stft(y=audio, sr=self.sr)
            # chroma는 12행이므로 n_mels 행으로 리사이즈
            chroma_resized = np.repeat(chroma, self.n_mels // 12 + 1, axis=0)[:self.n_mels, :]
            
            # 3채널 모두 같은 시간축 길이로 맞추기
            min_len = min(mel_db.shape[1], mfcc.shape[1], chroma_resized.shape[1])
            mel_db = mel_db[:, :min_len]
            mfcc = mfcc[:, :min_len]
            chroma_resized = chroma_resized[:, :min_len]
            current_length = min_len
        else:
            current_length = mel_db.shape[1]
        
        # 시간축 길이 맞추기 (패딩 / 크롭)
        if current_length < self.max_time_steps:
            pad_w = self.max_time_steps - current_length
            mel_db = np.pad(mel_db, ((0,0),(0,pad_w)), mode='constant', constant_values=-80)
            if self.n_channels == 3:
                mfcc = np.pad(mfcc, ((0,0),(0,pad_w)), mode='constant')
                chroma_resized = np.pad(chroma_resized, ((0,0),(0,pad_w)), mode='constant')
        elif current_length > self.max_time_steps:
            if self.is_train:
                s = random.randint(0, current_length - self.max_time_steps)
            else:
                s = (current_length - self.max_time_steps) // 2
            mel_db = mel_db[:, s:s+self.max_time_steps]
            if self.n_channels == 3:
                mfcc = mfcc[:, s:s+self.max_time_steps]
                chroma_resized = chroma_resized[:, s:s+self.max_time_steps]
        
        # 텐서 변환
        if self.n_channels == 3:
            # [3, n_mels, time] 컬러 이미지 형태
            tensor = torch.tensor(
                np.stack([mel_db, mfcc, chroma_resized], axis=0),
                dtype=torch.float32
            )
        else:
            # [1, n_mels, time] 흑백 이미지 형태
            tensor = torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0)
        
        label = torch.tensor(sample['label'], dtype=torch.float32)
        return tensor, label

print("공용 SpeakerDataset 클래스 준비 완료!")
print("  - n_channels=1: Mel-Spectrogram 흑백 (경로 B-1)")
print("  - n_channels=3: Mel+MFCC+Chroma 컬러 퓨전 (경로 B-2)")

In [ ]:
# ==========================================
# [경로 B] ResNet50 모델 정의 (1채널 / 3채널 자동 대응)
# ==========================================
class AudioResNet(nn.Module):
    def __init__(self, n_channels=1):
        super(AudioResNet, self).__init__()
        self.n_channels = n_channels
        self.model = models.resnet50(pretrained=True)
        
        if n_channels == 1:
            # 1채널: 기존 방식 — 3채널 가중치를 평균 내어 1채널로 압축 이식
            old_conv = self.model.conv1
            self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            self.model.conv1.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
        # n_channels == 3이면 순정 ResNet50 구조 그대로 사용! (개조 불필요)
        
        # 출력층: 1000가지 -> 1개 (이진 분류)
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, 1)
        
    def forward(self, x):
        return self.model(x)

print("AudioResNet 모델 정의 완료!")
print("  - AudioResNet(n_channels=1): 1채널 흑백 (Conv1 개조)")
print("  - AudioResNet(n_channels=3): 3채널 컬러 (순정 ResNet50)")

In [ ]:
# ==========================================
# [공용] 학습 + 평가 함수 (모든 CNN 모델이 공유)
# ==========================================
def train_and_evaluate(model, train_loader, val_loader, model_name, epochs=3, lr=1e-4):
    """
    모델을 학습시키고 검증 정확도를 반환합니다.
    반환값: (학습 시간, 검증 정확도, 검증 예측 확률 리스트)
    """
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    print(f"\n{'='*60}")
    print(f"  [{model_name}] 학습 시작 (Epochs={epochs}, LR={lr})")
    print(f"{'='*60}")
    
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for bx, by in bar:
            bx, by = bx.to(device), by.to(device).unsqueeze(1)
            optimizer.zero_grad()
            pred = model(bx)
            loss = criterion(pred, by)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            bar.set_postfix({'Loss': f'{loss.item():.4f}'})
    
    train_time = time.time() - t0
    
    # 검증
    model.eval()
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for bx, by in tqdm(val_loader, desc="검증 중"):
            bx = bx.to(device)
            probs = torch.sigmoid(model(bx)).cpu().numpy().flatten()
            all_probs.extend(probs)
            all_labels.extend(by.numpy().flatten())
    
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds = (all_probs >= 0.5).astype(int)
    accuracy = accuracy_score(all_labels, preds) * 100
    
    print(f"\n{'='*60}")
    print(f"  [{model_name}] 학습 시간: {train_time:.1f}초")
    print(f"  [{model_name}] 검증 정확도: {accuracy:.2f}%")
    print(f"{'='*60}")
    print(classification_report(all_labels, preds, target_names=['119대원(0)', '신고자(1)']))
    
    return train_time, accuracy, all_probs, all_labels

print("공용 학습/평가 함수 준비 완료!")

In [ ]:
# ==========================================
# [경로 B-1] ResNet50 1채널 학습 & 평가
# ==========================================
print("1채널 데이터셋 구성 중...")
train_ds_1ch = SpeakerDataset(train_samples, n_channels=1, is_train=True)
val_ds_1ch   = SpeakerDataset(val_samples,   n_channels=1, is_train=False)
train_dl_1ch = DataLoader(train_ds_1ch, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_dl_1ch   = DataLoader(val_ds_1ch,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model_1ch = AudioResNet(n_channels=1).to(device)
resnet1_time, resnet1_acc, resnet1_probs, val_labels = train_and_evaluate(
    model_1ch, train_dl_1ch, val_dl_1ch, "ResNet50 1ch (Baseline)", epochs=3
)

# 가중치 저장
torch.save(model_1ch.state_dict(), 'mission2_speaker/best_model_resnet1ch.pt')
print("ResNet50 1채널 가중치 저장 완료!")

---
# 🎨 경로 B-2: ResNet50 3채널 오디오 퓨전 (Mel + MFCC + Chroma)

R채널에 Mel-Spectrogram, G채널에 MFCC, B채널에 Chroma를 넣어
**순정 ResNet50(3채널 눈 그대로)**에 소리 컬러 사진을 입력합니다.
모델 구조를 개조할 필요가 없어 가장 안정적인 전이학습이 가능합니다.

In [ ]:
# ==========================================
# [경로 B-2] ResNet50 3채널 퓨전 학습 & 평가
# ==========================================
print("3채널 퓨전 데이터셋 구성 중...")
train_ds_3ch = SpeakerDataset(train_samples, n_channels=3, is_train=True)
val_ds_3ch   = SpeakerDataset(val_samples,   n_channels=3, is_train=False)
train_dl_3ch = DataLoader(train_ds_3ch, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_dl_3ch   = DataLoader(val_ds_3ch,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model_3ch = AudioResNet(n_channels=3).to(device)
resnet3_time, resnet3_acc, resnet3_probs, _ = train_and_evaluate(
    model_3ch, train_dl_3ch, val_dl_3ch, "ResNet50 3ch Fusion (Mel+MFCC+Chroma)", epochs=3
)

# 가중치 저장
torch.save(model_3ch.state_dict(), 'mission2_speaker/best_model_resnet3ch.pt')
print("ResNet50 3채널 퓨전 가중치 저장 완료!")

---
# 🎧 경로 C: Wav2Vec 2.0 (음성 특화 트랜스포머)

소리를 이미지로 변환하지 않고, **원본 파형(1D Waveform)을 그대로** 듣고 이해하는
메타(페이스북)의 최신 음성 특화 대형 모델입니다.

> ⚠️ VRAM을 약 7~8GB 이상 사용합니다. Colab Pro(L4/A100)에서 실행을 권장합니다.

In [ ]:
# ==========================================
# [경로 C] Wav2Vec 2.0 데이터셋 & 모델 정의
# ==========================================
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

# Wav2Vec2는 16kHz 오디오를 기대합니다
W2V_SR = 16000
W2V_MAX_LEN = W2V_SR * 3  # 3초 (48000 샘플)

class Wav2VecDataset(Dataset):
    """Wav2Vec2용 데이터셋: 원시 파형(1D)을 그대로 입력합니다."""
    def __init__(self, samples, sr=W2V_SR, max_len=W2V_MAX_LEN, is_train=True):
        self.samples = samples
        self.sr = sr
        self.max_len = max_len
        self.is_train = is_train
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        audio = cut_audio(sample, sr=self.sr)
        
        # 길이 맞추기 (패딩 / 크롭)
        if len(audio) < self.max_len:
            audio = np.pad(audio, (0, self.max_len - len(audio)), mode='constant')
        elif len(audio) > self.max_len:
            if self.is_train:
                s = random.randint(0, len(audio) - self.max_len)
            else:
                s = (len(audio) - self.max_len) // 2
            audio = audio[s:s+self.max_len]
        
        return torch.tensor(audio, dtype=torch.float32), torch.tensor(sample['label'], dtype=torch.long)

# HuggingFace에서 Wav2Vec2-base 모델 로드
print("Wav2Vec2-base 모델 로딩 중... (최초 실행 시 다운로드 ~360MB)")
w2v_model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=2,           # 이진 분류 (0: 대원, 1: 신고자)
    problem_type="single_label_classification"
)
w2v_model = w2v_model.to(device)
print(f"Wav2Vec2-base 로드 완료! 파라미터 수: {sum(p.numel() for p in w2v_model.parameters()):,}개")

In [ ]:
# ==========================================
# [경로 C] Wav2Vec 2.0 학습 & 평가
# ==========================================
W2V_BATCH = 16  # Wav2Vec2는 메모리를 많이 먹으므로 배치 크기를 줄임
W2V_EPOCHS = 3
W2V_LR = 5e-5   # 트랜스포머 모델은 학습률을 더 낮게 설정

train_ds_w2v = Wav2VecDataset(train_samples, is_train=True)
val_ds_w2v   = Wav2VecDataset(val_samples,   is_train=False)
train_dl_w2v = DataLoader(train_ds_w2v, batch_size=W2V_BATCH, shuffle=True,  num_workers=2)
val_dl_w2v   = DataLoader(val_ds_w2v,   batch_size=W2V_BATCH, shuffle=False, num_workers=2)

optimizer_w2v = torch.optim.AdamW(w2v_model.parameters(), lr=W2V_LR)

print(f"\n{'='*60}")
print(f"  [Wav2Vec2] 학습 시작 (Epochs={W2V_EPOCHS}, LR={W2V_LR})")
print(f"  VRAM 주의: 약 7~8GB 사용 (Colab L4/A100 권장)")
print(f"{'='*60}")

t0 = time.time()
for epoch in range(W2V_EPOCHS):
    w2v_model.train()
    bar = tqdm(train_dl_w2v, desc=f'W2V Epoch {epoch+1}/{W2V_EPOCHS}')
    for bx, by in bar:
        bx, by = bx.to(device), by.to(device)
        optimizer_w2v.zero_grad()
        outputs = w2v_model(input_values=bx, labels=by)
        loss = outputs.loss
        loss.backward()
        optimizer_w2v.step()
        bar.set_postfix({'Loss': f'{loss.item():.4f}'})

w2v_train_time = time.time() - t0

# 검증
w2v_model.eval()
w2v_all_probs = []
w2v_all_labels = []
with torch.no_grad():
    for bx, by in tqdm(val_dl_w2v, desc="W2V 검증 중"):
        bx = bx.to(device)
        outputs = w2v_model(input_values=bx)
        # softmax로 확률 변환 (클래스 1 = 신고자 확률)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
        w2v_all_probs.extend(probs)
        w2v_all_labels.extend(by.numpy())

w2v_all_probs = np.array(w2v_all_probs)
w2v_all_labels = np.array(w2v_all_labels)
w2v_preds = (w2v_all_probs >= 0.5).astype(int)
w2v_accuracy = accuracy_score(w2v_all_labels, w2v_preds) * 100

print(f"\n{'='*60}")
print(f"  [Wav2Vec2] 학습 시간: {w2v_train_time:.1f}초")
print(f"  [Wav2Vec2] 검증 정확도: {w2v_accuracy:.2f}%")
print(f"{'='*60}")
print(classification_report(w2v_all_labels, w2v_preds, target_names=['119대원(0)', '신고자(1)']))

# 가중치 저장
torch.save(w2v_model.state_dict(), 'mission2_speaker/best_model_wav2vec2.pt')
print("Wav2Vec2 가중치 저장 완료!")

---
# 📊 종합 비교 분석 & 하이브리드 앙상블

모든 모델의 성능을 한눈에 비교하고, **시각파(CNN)와 청각파(Transformer)의 확률을 결합**하여
최종 앙상블 정확도를 도출합니다.

In [ ]:
# ==========================================
# 종합 비교 리포트 생성
# ==========================================
results = {
    'RandomForest':          {'accuracy': rf_accuracy,    'train_time': rf_train_time,    'params': '~200 trees'},
    'ResNet50 1ch (기존)':    {'accuracy': resnet1_acc,    'train_time': resnet1_time,     'params': '23.5M'},
    'ResNet50 3ch Fusion':   {'accuracy': resnet3_acc,    'train_time': resnet3_time,     'params': '23.5M'},
    'Wav2Vec2-base':         {'accuracy': w2v_accuracy,   'train_time': w2v_train_time,   'params': '94.4M'},
}

print("\n" + "="*70)
print("  [Mission 2] 종합 모델 비교 리포트")
print("="*70)
print(f"{'모델':<25} {'정확도':>10} {'학습시간':>10} {'파라미터':>10}")
print("-"*70)
for name, r in results.items():
    print(f"{name:<25} {r['accuracy']:>8.2f}%  {r['train_time']:>8.1f}s  {str(r['params']):>10}")
print("="*70)

In [ ]:
# ==========================================
# 하이브리드 앙상블 (Soft Voting)
# ==========================================
# 최적의 혼합 비율(alpha) 탐색
print("\n하이브리드 앙상블 최적 비율 탐색 중...")
best_alpha = 0
best_ensemble_acc = 0

for alpha_pct in range(0, 101, 5):  # 0%, 5%, 10%, ..., 100%
    alpha = alpha_pct / 100.0
    # ResNet50 3ch 확률과 Wav2Vec2 확률을 혼합
    ensemble_probs = alpha * resnet3_probs + (1 - alpha) * w2v_all_probs
    ensemble_preds = (ensemble_probs >= 0.5).astype(int)
    acc = accuracy_score(val_labels, ensemble_preds) * 100
    
    if acc > best_ensemble_acc:
        best_ensemble_acc = acc
        best_alpha = alpha

print(f"\n{'='*60}")
print(f"  [하이브리드 앙상블] 최적 혼합 비율: ResNet3ch {best_alpha:.0%} + Wav2Vec2 {1-best_alpha:.0%}")
print(f"  [하이브리드 앙상블] 최종 검증 정확도: {best_ensemble_acc:.2f}%")
print(f"{'='*60}")

# 최종 비교 결과에 앙상블 추가
results['Ensemble (최종)'] = {
    'accuracy': best_ensemble_acc,
    'train_time': resnet3_time + w2v_train_time,
    'params': f'alpha={best_alpha:.2f}'
}

In [ ]:
# ==========================================
# 시각화: 모델별 정확도 비교 차트
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1) 정확도 비교 막대 그래프
names = list(results.keys())
accs = [results[n]['accuracy'] for n in names]
colors = ['#95a5a6', '#3498db', '#2ecc71', '#e74c3c', '#f39c12']
bars = axes[0].barh(names, accs, color=colors[:len(names)], edgecolor='white', height=0.6)
axes[0].set_xlabel('검증 정확도 (%)', fontsize=12)
axes[0].set_title('Mission 2 모델별 정확도 비교', fontsize=14, fontweight='bold')
axes[0].set_xlim(70, 100)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{acc:.2f}%', va='center', fontsize=11, fontweight='bold')

# 2) 학습 시간 비교
times = [results[n]['train_time'] for n in names]
axes[1].barh(names, times, color=colors[:len(names)], edgecolor='white', height=0.6)
axes[1].set_xlabel('학습 시간 (초)', fontsize=12)
axes[1].set_title('모델별 학습 시간 비교', fontsize=14, fontweight='bold')
for i, t in enumerate(times):
    axes[1].text(t + max(times)*0.02, i, f'{t:.0f}s', va='center', fontsize=11)

plt.tight_layout()
plt.savefig('mission2_speaker/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("비교 차트 저장 완료! (mission2_speaker/model_comparison.png)")

In [ ]:
# ==========================================
# 비교 결과 JSON 저장 (PPT/보고서용)
# ==========================================
import json as json_mod

comparison_data = {}
for name, r in results.items():
    comparison_data[name] = {
        'accuracy_pct': round(r['accuracy'], 2),
        'train_time_sec': round(r['train_time'], 1),
        'params': str(r['params'])
    }

with open('mission2_speaker/comparison.json', 'w', encoding='utf-8') as f:
    json_mod.dump(comparison_data, f, ensure_ascii=False, indent=2)

print("비교 결과 JSON 저장 완료! (mission2_speaker/comparison.json)")
print("\n=== 전체 실험 완료! ===")
print(f"최종 추천 제출 모델: 하이브리드 앙상블 (정확도 {best_ensemble_acc:.2f}%)")

---
# 🚀 [최신] 전체 데이터 5대 모델 벤치마크 자동 실행
> **수정 완료**: Windows 호환 패치, AMP 혼합정밀도, 윈도우 프리징 방지(num_workers=0)
> 전체 데이터 87만 개로 ResNet-50, ECAPA-TDNN, ReDimNet, CAM++, SSAST를 순차 학습합니다.


In [ ]:
import os, sys
os.chdir(r"C:\Users\user\Desktop\DCC\mission2_speaker")
if "." not in sys.path:
    sys.path.insert(0, ".")

from benchmark_suite.benchmark import run_benchmark

# 전체 5대 모델 벤치마크 학습 실행
run_benchmark(
    models_to_run=["resnet50", "ecapa_tdnn", "redimnet", "campp", "ssast"],
    epochs=10,
    max_train_files=None,   # 전체 데이터 100%
    max_val_files=None,
    data_root=r"C:\Users\user\Desktop\DCC\data",
    output_dir="./results",
    skip_completed=True
)



[벤치마크 시작] 총 5개 모델 테스트 예정: ['resnet50', 'ecapa_tdnn', 'redimnet', 'campp', 'ssast']
데이터 경로: C:\Users\user\Desktop\DCC\data
결과 저장 경로: ./results


[모델 벤치마크 실행] : RESNET50


  Training:   0%|          | 0/13642 [00:00<?, ?it/s]

[resnet50] Ep 01/10 | Tr Loss: 0.6945 Tr Acc: 51.21% | Val Loss: nan Val Acc: 51.92% F1: 0.3421


  Training:   0%|          | 0/13642 [00:00<?, ?it/s]

[resnet50] Ep 02/10 | Tr Loss: 0.6932 Tr Acc: 51.53% | Val Loss: nan Val Acc: 48.10% F1: 0.3248


...

  Training:   0%|          | 0/13642 [00:00<?, ?it/s]

[resnet50] Ep 05/10 | Tr Loss: 0.6926 Tr Acc: 51.94% | Val Loss: nan Val Acc: 51.92% F1: 0.3421


  Training:   0%|          | 0/13642 [00:00<?, ?it/s]

[resnet50] Ep 06/10 | Tr Loss: 0.6925 Tr Acc: 51.97% | Val Loss: nan Val Acc: 51.92% F1: 0.3421


  Training:   0%|          | 0/13642 [00:00<?, ?it/s]